-- Job ID: rmisamplesql_traffic_YYYYMMDDHHMMSS
```
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
```


# RMI Sample Queries: Traffic Operations Manager (GA)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Traffic_Operations_Manager_Samples.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Frmi_sample_queries%2Fnotebooks%2FTraffic_Operations_Manager_Samples.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Traffic_Operations_Manager_Samples.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/rmi_sample_queries/notebooks/Traffic_Operations_Manager_Samples.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>



This notebook contains sample queries for the **Traffic Operations Manager** persona, specifically for the **GA** stage.

## 1. Setup

In [ ]:
from google.colab import auth
import pandas as pd

auth.authenticate_user()

project_id = 'your-project-id' #@param {type:"string"}

### Writable Dataset

Several queries in this notebook (e.g., those creating Materialized Views, Models, or Views) require a **writable dataset** within your own project. 
**Note**: The source `LINKED_DATASET_NAME` dataset is a read-only subscription and cannot be used to store new resources.

Run the cell below to create a new dataset (e.g., `rmi_analysis`) in your project if you haven't already.

**Important**: When running queries that create new BigQuery resources (e.g., tables, views, models) outside of these `%%bigquery` magic cells, remember to manually prepend the job ID with `msqlfactory--` for proper tracking. For example: `bq query --job_id=msqlfactory--your-descriptive-job-name ...`

In [ ]:
dataset_id = "rmi_analysis" #@param {type:"string"}
! bq --location=US mk --dataset {project_id}:{dataset_id}

## GA (General Availability)

### tom1_peak_hour_delay.sql
**Business Question**: What is the average travel time delay during the morning peak (7-9 AM) for the top 10 most congested routes, explicitly accounting for road closures?
**Use Case**: Identifies critical morning commute bottlenecks and closure incidents to inform operational decisions or public messaging.


In [ ]:
%%bigquery --project {project_id} df_tom1_peak_hour_delay
/*
  ANALYTICAL PATTERN: Temporal Filtering & Closure Safety
  This query uses EXTRACT(HOUR...) on a converted DATETIME to focus on local 
  peak windows. It joins routes_status on s.status = 'STATUS_RUNNING' and
  explicitly tracks has_road_closure rather than silently dropping closures
  with a naive duration_in_seconds IS NOT NULL filter.
*/

WITH peak_hour_data AS (
  SELECT
    h.selected_route_id,
    h.display_name,
    h.has_road_closure,
    -- delay_ratio > 1.0 indicates travel time is slower than free-flow (static)
    SAFE_DIVIDE(h.duration_in_seconds, h.static_duration_in_seconds) AS delay_ratio
  FROM `LINKED_DATASET_NAME.historical_travel_time` AS h
  JOIN `LINKED_DATASET_NAME.routes_status` AS s USING (selected_route_id)
  WHERE h.record_time BETWEEN '2026-07-01' AND '2026-07-30'
    -- STATUS_RUNNING ensures we only analyze routes that are currently being monitored
    AND s.status = 'STATUS_RUNNING'
    -- AM Peak Window: 7:00 AM to 8:59 AM Local Time
    AND EXTRACT(HOUR FROM DATETIME(h.record_time, 'America/New_York')) BETWEEN 7 AND 8
    -- Geometry Integrity: Only process continuous, healthy paths (ST_LineString)
    AND ST_GEOMETRYTYPE(h.route_geometry) = 'ST_LineString'
    AND (h.static_duration_in_seconds > 0 OR h.has_road_closure IS TRUE)
)
SELECT
  display_name,
  -- Compute average delay ratio on passable periods
  ROUND(AVG(CASE WHEN COALESCE(has_road_closure, FALSE) IS FALSE THEN delay_ratio END), 3) AS avg_delay_ratio,
  COUNT(*) AS total_observations,
  -- Explicitly audit road closure incidents rather than silently dropping them
  COUNTIF(has_road_closure IS TRUE) AS road_closure_incidents,
  COUNTIF(COALESCE(has_road_closure, FALSE) IS FALSE AND delay_ratio IS NOT NULL) AS valid_delay_samples
FROM peak_hour_data
GROUP BY 1
-- Threshold: Filter for routes that are slower than free-flow or experienced road closures
HAVING avg_delay_ratio > 1.0 OR road_closure_incidents > 0
ORDER BY road_closure_incidents DESC, avg_delay_ratio DESC
LIMIT 10;


### tom2_persistent_bottlenecks.sql
**Business Question**: Which routes and segments suffer from persistent traffic jams (TRAFFIC_JAM / TRAFFIC_JAM_HEAVY), and what is the physical distance (meters/km) in congestion?
**Use Case**: Locates recurring local bottlenecks within monitored corridors, quantifying both standstill gridlock and standard congestion mileage to guide operational intervention.


In [ ]:
%%bigquery --project {project_id} df_tom2_persistent_bottlenecks
/*
  ANALYTICAL PATTERN: SRI Unnesting with Modern Offsets
  RMI routes store segment-level traffic states in speed_reading_offsets.
  This query unpacks that array to audit the frequency and physical length
  of severe congestion (TRAFFIC_JAM and TRAFFIC_JAM_HEAVY) using pure integer arithmetic.
*/

WITH congested_offsets AS (
  SELECT
    r.selected_route_id,
    s.display_name,
    r.record_time,
    sro.speed,
    -- Compute physical interval length in meters via integer offset subtraction
    (sro.end_offset - sro.start_offset) AS interval_length_meters
  FROM `LINKED_DATASET_NAME.recent_roads_data` AS r
  JOIN `LINKED_DATASET_NAME.routes_status` AS s USING (selected_route_id)
  LEFT JOIN UNNEST(r.speed_reading_offsets) AS sro
  WHERE r.record_time BETWEEN '2026-07-01' AND '2026-07-30'
    AND s.status = 'STATUS_RUNNING'
    AND (r.has_road_closure IS FALSE OR r.has_road_closure IS NULL)
    AND ST_GEOMETRYTYPE(r.route_geometry) = 'ST_LineString'
    -- Focus exclusively on severe and standard traffic congestion tiers
    AND sro.speed IN ('TRAFFIC_JAM', 'TRAFFIC_JAM_HEAVY')
)
SELECT
  selected_route_id,
  display_name,
  -- Count total congestion interval occurrences
  COUNT(*) AS total_congested_intervals,
  -- Break out counts by congestion severity
  COUNTIF(speed = 'TRAFFIC_JAM_HEAVY') AS heavy_jam_occurrences,
  COUNTIF(speed = 'TRAFFIC_JAM') AS standard_jam_occurrences,
  -- Aggregate physical distance in congestion (km)
  ROUND(SUM(interval_length_meters) / 1000.0, 2) AS total_congested_km,
  ROUND(SUM(CASE WHEN speed = 'TRAFFIC_JAM_HEAVY' THEN interval_length_meters ELSE 0 END) / 1000.0, 2) AS heavy_jam_km,
  ROUND(AVG(interval_length_meters), 1) AS avg_congested_interval_meters
FROM congested_offsets
GROUP BY 1, 2
ORDER BY heavy_jam_km DESC, total_congested_km DESC
LIMIT 10;


### tom3_operational_health.sql
**Business Question**: Which active routes are currently flagged with a 'LOW_ROAD_USAGE' validation error?
**Use Case**: Monitors the reliability of data collection. Low usage flags indicate that insights for these routes may be based on fewer probes, requiring a review of route priority or placement.


In [ ]:
%%bigquery --project {project_id} df_tom3_operational_health
/*
  ANALYTICAL PATTERN: Status Auditing
  This query inspects the management plane table (routes_status) to identify 
  active routes that have quality warnings. This is critical for maintaining 
  trust in downstream traffic analytics.
*/

SELECT
  display_name,
  selected_route_id,
  status,
  validation_error,
  -- 'low_road_usage_start_time' is specifically populated when probe density drops below threshold
  low_road_usage_start_time,
  -- Time elapsed since the error was first detected (relative to sample end date)
  DATETIME_DIFF(DATETIME('2026-07-30'), DATETIME(low_road_usage_start_time, 'UTC'), DAY) AS days_in_error_state
FROM `LINKED_DATASET_NAME.routes_status`
-- We only care about errors on routes that are supposed to be active (STATUS_RUNNING)
WHERE status = 'STATUS_RUNNING'
  -- Filter specifically for the Low Road Usage warning
  AND validation_error = 'VALIDATION_ERROR_LOW_ROAD_USAGE'
ORDER BY days_in_error_state DESC;


### tom4_data_latency.sql
**Business Question**: Are there any active routes that have stopped sending data near the end of the snapshot period?
**Use Case**: Detects localized data gaps or "silent" routes in real-time, enabling operators to investigate issues before they impact reporting.


In [ ]:
%%bigquery --project {project_id} df_tom4_data_latency
/*
  ANALYTICAL PATTERN: Freshness Monitoring
  By comparing the max record_time per route against the overall dataset 
  end-time, we can identify routes that have 'gone silent'.
*/

WITH last_data_arrival AS (
  SELECT
    selected_route_id,
    -- Get the latest record timestamp for every route in the dataset
    MAX(record_time) AS last_arrival
  FROM `LINKED_DATASET_NAME.historical_travel_time`
  -- Focused partition scan for the full sample month
  WHERE ST_GEOMETRYTYPE(route_geometry) = 'ST_LineString' AND record_time BETWEEN '2026-07-01' AND '2026-07-30'
  GROUP BY 1
)
SELECT
  s.selected_route_id,
  s.display_name,
  l.last_arrival,
  -- Measured relative to the end of the sample period
  TIMESTAMP_DIFF(TIMESTAMP('2026-07-30 00:00:00'), l.last_arrival, MINUTE) as minutes_of_silence
FROM `LINKED_DATASET_NAME.routes_status` s
LEFT JOIN last_data_arrival l USING (selected_route_id)
-- Focus on routes that are supposed to be producing data
WHERE s.status = 'STATUS_RUNNING'
  -- Threshold: Highlight routes that haven't sent a record in the last few hours of the dataset
  AND l.last_arrival < TIMESTAMP('2026-07-29 21:55:00')
ORDER BY minutes_of_silence DESC;


### tom5_significant_event_detection.sql
**Business Question**: Which routes experienced a significant event (full road closure OR travel time more than double static baseline) in the last 24-72 hours?
**Use Case**: Automates the detection of extreme traffic events (accidents, road closures, severe gridlock) requiring immediate operational intervention.


In [ ]:
%%bigquery --project {project_id} df_tom5_significant_event_detection
/*
  ANALYTICAL PATTERN: Threshold-Based Alerting with Road Closure Branch
  This query identifies major disruptions by flagging:
  1. Full road closures (has_road_closure = true, duration_in_seconds IS NULL).
  2. Severe congestion events where actual travel time is at least 2x the free-flow baseline.
*/

SELECT
  h.display_name,
  h.selected_route_id,
  h.record_time,
  h.has_road_closure,
  h.duration_in_seconds,
  h.static_duration_in_seconds,
  -- Classify the event type
  CASE
    WHEN h.has_road_closure IS TRUE THEN 'FULL_ROAD_CLOSURE'
    WHEN SAFE_DIVIDE(h.duration_in_seconds, h.static_duration_in_seconds) > 2.0 THEN 'SEVERE_CONGESTION'
    ELSE 'NOMINAL'
  END AS event_type,
  -- Delay ratio > 2.0 means travel time is 2x slower than ideal (NULL for full closures)
  ROUND(SAFE_DIVIDE(h.duration_in_seconds, h.static_duration_in_seconds), 2) AS delay_ratio
FROM `LINKED_DATASET_NAME.historical_travel_time` AS h
JOIN `LINKED_DATASET_NAME.routes_status` AS s USING (selected_route_id)
WHERE h.record_time BETWEEN '2026-07-27' AND '2026-07-30'
  AND s.status = 'STATUS_RUNNING'
  -- Trigger alert if road is completely closed OR travel time exceeds 2x baseline
  AND (h.has_road_closure IS TRUE OR SAFE_DIVIDE(h.duration_in_seconds, h.static_duration_in_seconds) > 2.0)
  -- Quality filter: Exclude non-continuous geometries
  AND ST_GEOMETRYTYPE(h.route_geometry) = 'ST_LineString'
  AND (h.static_duration_in_seconds > 0 OR h.has_road_closure IS TRUE)
ORDER BY h.has_road_closure DESC, delay_ratio DESC;


### tom6_dynamic_detour_detection.sql
**Business Question**: Which dynamic routes experienced rerouting/detours away from their baseline static sequence, and what was the travel time impact between baseline and detour variants?
**Use Case**: Identifies dynamic route variations and incident diversions on dynamic_routing = true corridors by comparing observed segment sequences against routes_status.road_segment_ids.


In [ ]:
%%bigquery --project {project_id} df_tom6_dynamic_detour_detection
/*
  ANALYTICAL PATTERN: Dynamic Detour Verification via road_segment_ids
  
  Leverages:
  1. s.dynamic_routing = true: Focuses on corridors configured for dynamic rerouting.
  2. s.road_segment_ids: Authoritative static baseline Place ID sequence from routes_status.
  3. h.road_segment_ids: Actual observed traffic-aware Place ID sequence (available since 2026-07-01).
  4. Detour Delay Multiplier: Compares detour average duration against baseline path duration.
*/

WITH path_variants AS (
  SELECT
    h.selected_route_id,
    h.display_name,
    s.dynamic_routing,
    ARRAY_TO_STRING(s.road_segment_ids, '|') AS baseline_segment_ids,
    ARRAY_TO_STRING(h.road_segment_ids, '|') AS observed_segment_ids,
    -- Check if observed path matches registered static baseline
    (ARRAY_TO_STRING(h.road_segment_ids, '|') = ARRAY_TO_STRING(s.road_segment_ids, '|')) AS is_baseline_path,
    AVG(h.duration_in_seconds) AS avg_duration_in_seconds,
    COUNT(h.record_time) AS num_of_records
  FROM `LINKED_DATASET_NAME.historical_travel_time` AS h
  JOIN `LINKED_DATASET_NAME.routes_status` AS s USING (selected_route_id)
  -- Schema Temporal Filter: road_segment_ids is only populated from 2026-07-01 onward
  WHERE h.record_time >= '2026-07-01'
    AND ARRAY_LENGTH(h.road_segment_ids) > 0
    AND ARRAY_LENGTH(s.road_segment_ids) > 0
    AND s.status = 'STATUS_RUNNING'
    -- Focus strictly on routes configured for dynamic traffic rerouting
    AND s.dynamic_routing IS TRUE
    AND (h.has_road_closure IS FALSE OR h.has_road_closure IS NULL)
    AND h.duration_in_seconds IS NOT NULL
    AND h.static_duration_in_seconds > 0
  GROUP BY 1, 2, 3, 4, 5, 6
),
ranked_routes AS (
  SELECT
    selected_route_id,
    display_name,
    dynamic_routing,
    baseline_segment_ids,
    ARRAY_AGG(
      STRUCT(observed_segment_ids, is_baseline_path, avg_duration_in_seconds, num_of_records)
      ORDER BY is_baseline_path DESC, num_of_records DESC
    ) AS variations
  FROM path_variants
  GROUP BY 1, 2, 3, 4
)
SELECT
  selected_route_id,
  display_name,
  ARRAY_LENGTH(variations) AS total_path_variations,
  variations[OFFSET(0)].num_of_records AS baseline_path_samples,
  ROUND(variations[OFFSET(0)].avg_duration_in_seconds, 1) AS baseline_avg_duration_seconds,
  variations[OFFSET(1)].num_of_records AS detour_path_samples,
  ROUND(variations[OFFSET(1)].avg_duration_in_seconds, 1) AS detour_avg_duration_seconds,
  -- Detour Delay Multiplier: > 1.0 means the detour was slower than the baseline path
  ROUND(SAFE_DIVIDE(variations[OFFSET(1)].avg_duration_in_seconds, variations[OFFSET(0)].avg_duration_in_seconds), 2) AS detour_delay_multiplier
FROM ranked_routes
WHERE ARRAY_LENGTH(variations) > 1
ORDER BY detour_delay_multiplier DESC;
